# SciCite-REAL — demo
Retrieval-Augmented Citation Intent Classification on SciCite.

This notebook runs the full pipeline on a tiny slice so you can see the BM25 evidence materialize before committing to a full GPU run.

In [ ]:
import pickle, sys, json
from pathlib import Path

sys.path.append(".")
from src.data.dataset import load_scicite, read_jsonl, stat_summary
from src.retrieval.index import BM25Index
from src.retrieval.retrieval_augmented import RetrievalAugmenter

In [ ]:
# 1) quick pointer to data (already preprocessed)
DATA = Path("data/processed/scicite")
assert DATA.exists(), "run `python scripts/preprocess.py` first"

In [ ]:
# 2) build a mini BM25 index over the first 200 train contexts
rows = read_jsonl(DATA / "train.jsonl")[:200]
index = BM25Index([r["string"] for r in rows], variant="bm25l")
index.labels = [r["label"] for r in rows]
print("mini index ready:", len(rows), "docs")

In [ ]:
# 3) augment one real test example and inspect the evidence
test_rows = read_jsonl(DATA / "test.jsonl")[:5]
aug = RetrievalAugmenter(index=index, top_k=3, leak_free=False, labels=["background","method","result"])

for i, ex in enumerate(test_rows):
    r = aug.augment(ex["string"], ex["label"], doc_id=ex["id"])
    print(f"\n=== test example #{i} (true label: {ex['label']}) ===")
    print("TARGET        :", ex["string"][:140], "...")
    for j, ev in enumerate(r["evidence"]):
        print(f"  BM25 evidence{j}: {ev[:120]}...")

In [ ]:
# 4) tokenize the augmented text ready for training
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")
enc = tok(r["text"], truncation=True, max_length=256, return_tensors="pt")
print("tokens:", enc["input_ids"].shape, "| input begins:", enc["input_ids"][0][:8].tolist())